# 07 Ablation Study

Train the four controlled ablation experiments after Notebook 06 has selected a candidate architecture. This notebook keeps the validation and test splits fixed, changes only the training recipe, and ranks experiments using validation metrics only.


## Purpose

The goal is to isolate the value of online augmentation, offline augmentation, and the full pipeline. Every experiment uses the same selected YOLO architecture so the comparison reflects the data and augmentation strategy rather than model capacity.


## 1. Setup

This cell finds the `v2_pipeline` root from the notebook location, adds it to `sys.path`, and sets Windows/Jupyter safety environment variables before importing the training helpers.


In [1]:
from pathlib import Path
import os
import sys

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")


def find_v2_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "configs").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not locate v2_pipeline root.")


try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd().resolve()

V2_ROOT = find_v2_root(NOTEBOOK_DIR)
if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))

print(f"v2_pipeline root: {V2_ROOT}")

v2_pipeline root: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline


The next cell imports project helpers and third-party packages. Keeping imports here makes dependency errors easy to spot before any training run starts.


In [2]:
import pandas as pd
import yaml

from src.training.train_yolo import train_ablation_experiments
from src.training.evaluate_yolo import benchmark_model_latency, rank_ablation_results

## 2. Load Configuration

Notebook 07 reads the same project config files as the earlier notebooks. Online augmentation values are passed as Ultralytics training arguments only for experiments B and D; they are not written into dataset YAML files.


In [3]:
CONFIG_DIR = V2_ROOT / "configs"

with (CONFIG_DIR / "training_config.yaml").open("r", encoding="utf-8") as file_handle:
    training_config = yaml.safe_load(file_handle)

with (CONFIG_DIR / "augmentation_config.yaml").open(
    "r", encoding="utf-8"
) as file_handle:
    augmentation_config = yaml.safe_load(file_handle)

with (CONFIG_DIR / "class_names.yaml").open("r", encoding="utf-8") as file_handle:
    class_config = yaml.safe_load(file_handle)

online_aug = augmentation_config.get("online_augmentation", {})
class_config

{'nc': 3, 'names': {0: 'person', 1: 'helmet', 2: 'vest'}}

## 3. Locate Experiment Datasets

The ablation datasets must already exist from Notebook 05. This notebook only reads the four dataset YAML files and never rebuilds or modifies the experiment folders.


In [4]:
EXPERIMENT_YAMLS = {
    "exp_A_original_only": V2_ROOT / "data_exp_A_original_only.yaml",
    "exp_B_online_aug": V2_ROOT / "data_exp_B_online_aug.yaml",
    "exp_C_offline_aug": V2_ROOT / "data_exp_C_offline_aug.yaml",
    "exp_D_full_pipeline": V2_ROOT / "data_exp_D_full_pipeline.yaml",
}

missing_yamls = [str(path) for path in EXPERIMENT_YAMLS.values() if not path.exists()]
if missing_yamls:
    raise FileNotFoundError(
        "Missing experiment YAML files from Notebook 05: " + ", ".join(missing_yamls)
    )

rows = []
for experiment, yaml_path in EXPERIMENT_YAMLS.items():
    with yaml_path.open("r", encoding="utf-8") as file_handle:
        data_config = yaml.safe_load(file_handle)
    dataset_root = (yaml_path.parent / data_config["path"]).resolve()
    rows.append(
        {
            "experiment": experiment,
            "yaml": str(yaml_path.relative_to(V2_ROOT)),
            "dataset_root_exists": dataset_root.exists(),
            "train_images_exists": (dataset_root / data_config["train"]).exists(),
            "val_images_exists": (dataset_root / data_config["val"]).exists(),
            "test_images_exists": (dataset_root / data_config["test"]).exists(),
        }
    )

dataset_check_df = pd.DataFrame(rows)
display(dataset_check_df)

if (
    not dataset_check_df[
        [
            "dataset_root_exists",
            "train_images_exists",
            "val_images_exists",
            "test_images_exists",
        ]
    ]
    .all()
    .all()
):
    raise FileNotFoundError(
        "One or more ablation dataset folders are missing. Re-run Notebook 05 first."
    )

,experiment,yaml,dataset_root_exists,train_images_exists,val_images_exists,test_images_exists
0,exp_A_original_only,data_exp_A_original_only.yaml,True,True,True,True
1,exp_B_online_aug,data_exp_B_online_aug.yaml,True,True,True,True
2,exp_C_offline_aug,data_exp_C_offline_aug.yaml,True,True,True,True
3,exp_D_full_pipeline,data_exp_D_full_pipeline.yaml,True,True,True,True


## 4. Resolve Selected Architecture

Ablation must use one fixed architecture. The preferred source is the ranked candidate report from Notebook 06. If that report is unavailable, the notebook falls back to `selected_model` in `training_config.yaml`, then to the first configured candidate model.


In [5]:
REPORTS_DIR = V2_ROOT / "reports" / "training"
CANDIDATE_RANKING_CSV = REPORTS_DIR / "candidate_model_ranking.csv"

selected_model = training_config.get("selected_model")
selection_source = "training_config.selected_model"

if CANDIDATE_RANKING_CSV.exists():
    candidate_ranking = pd.read_csv(CANDIDATE_RANKING_CSV)
    if "status" in candidate_ranking.columns:
        trained_candidates = candidate_ranking[
            candidate_ranking["status"].eq("trained")
        ]
    else:
        trained_candidates = candidate_ranking
    if not trained_candidates.empty and "model_name" in trained_candidates.columns:
        selected_model = trained_candidates.iloc[0]["model_name"]
        selection_source = "candidate_model_ranking.csv"

if not selected_model:
    candidate_models = training_config.get("candidate_models", [])
    if not candidate_models:
        raise ValueError(
            "No selected model or candidate_models found in training_config.yaml."
        )
    selected_model = candidate_models[0]
    selection_source = "first configured candidate model fallback"

print(f"Selected architecture: {selected_model}")
print(f"Selection source: {selection_source}")

Selected architecture: yolov8n.pt
Selection source: candidate_model_ranking.csv


## 5. Build Training Arguments

Experiments A and C disable online augmentation. Experiments B and D receive the configured online augmentation values. All experiments share the same model, epochs, image size, batch size, device, workers, patience, seed, and AMP setting.


In [6]:
ABLATION_EPOCHS = int(
    training_config.get(
        "ablation_epochs",
        training_config.get("final_epochs", training_config["epochs"]),
    )
)
DRY_RUN = bool(training_config.get("dry_run_ablation", False))
OVERWRITE_EXISTING_RUNS = bool(training_config.get("overwrite_ablation_runs", False))

base_train_args = {
    "epochs": ABLATION_EPOCHS,
    "imgsz": int(training_config["imgsz"]),
    "batch": int(training_config["batch"]),
    "device": training_config["device"],
    "workers": int(training_config["workers"]),
    "patience": int(training_config["patience"]),
    "seed": int(training_config["seed"]),
    "amp": bool(training_config.get("amp", False)),
    "dry_run": DRY_RUN,
    "overwrite": OVERWRITE_EXISTING_RUNS,
}

online_train_args = {
    "hsv_h": float(online_aug.get("hsv_h", 0.0)),
    "hsv_s": float(online_aug.get("hsv_s", 0.0)),
    "hsv_v": float(online_aug.get("hsv_v", 0.0)),
    "degrees": float(online_aug.get("degrees", 0.0)),
    "translate": float(online_aug.get("translate", 0.0)),
    "scale": float(online_aug.get("scale", 0.0)),
    "perspective": float(online_aug.get("perspective", 0.0)),
    "fliplr": float(online_aug.get("fliplr", 0.0)),
    "flipud": float(online_aug.get("flipud", 0.0)),
    "mosaic": float(online_aug.get("mosaic", 0.0)),
    "mixup": float(online_aug.get("mixup", 0.0)),
    "close_mosaic": int(online_aug.get("close_mosaic", 0)),
}

strategy_df = pd.DataFrame(
    [
        {
            "experiment": "exp_A_original_only",
            "train_data": "original train",
            "online_augmentation": "off",
        },
        {
            "experiment": "exp_B_online_aug",
            "train_data": "original train",
            "online_augmentation": "on",
        },
        {
            "experiment": "exp_C_offline_aug",
            "train_data": "original + offline augmented train",
            "online_augmentation": "off",
        },
        {
            "experiment": "exp_D_full_pipeline",
            "train_data": "original + offline augmented train",
            "online_augmentation": "on",
        },
    ]
)

display(strategy_df)
print("Shared training args:")
display(pd.DataFrame([base_train_args]))
print("Online augmentation args for B and D:")
display(pd.DataFrame([online_train_args]))

,experiment,train_data,online_augmentation
0,exp_A_original_only,original train,off
1,exp_B_online_aug,original train,on
2,exp_C_offline_aug,original + offline augmented train,off
3,exp_D_full_pipeline,original + offline augmented train,on


Shared training args:


,epochs,imgsz,batch,device,workers,patience,seed,amp,dry_run,overwrite
0,100,640,24,0,8,20,42,True,False,False


Online augmentation args for B and D:


,hsv_h,hsv_s,hsv_v,degrees,translate,scale,perspective,fliplr,flipud,mosaic,mixup,close_mosaic
0,0.015,0.3,0.35,5.0,0.1,0.5,0.0003,0.5,0.0,1.0,0.05,10


## 6. Train Ablation Experiments

This cell launches the controlled ablation runs under `runs/ablation/`. If a run fails, the error is recorded and the remaining experiments continue.


In [7]:
ABLATION_RUNS_DIR = V2_ROOT / "runs" / "ablation"

ablation_results = train_ablation_experiments(
    experiment_data_yamls=EXPERIMENT_YAMLS,
    model_name=selected_model,
    output_dir=ABLATION_RUNS_DIR,
    base_train_args=base_train_args,
    online_augmentation_args=online_train_args,
    continue_on_error=True,
)

display(ablation_results)

New https://pypi.org/project/ultralytics/8.4.58 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.51  Python-3.10.20 torch-2.13.0.dev20260517+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16311MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=24, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\runs\ablation\_data_exp_A_original_only_resolved.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.0

,model_name,run_name,status,run_dir,precision,recall,map50,map50_95,fitness,training_time_seconds,best_weights_path,last_weights_path,model_size_mb,error_message,notes,experiment,data_yaml,resolved_data_yaml,uses_online_augmentation
0,yolov8n.pt,yolov8n_A_original_only,trained,C:\Github\smart-factory-safety-monitoring-syst...,0.90231,0.77549,0.83867,0.55291,None,128.637,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.952494,,,exp_A_original_only,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,False
1,yolov8n.pt,yolov8n_B_online_aug,trained,C:\Github\smart-factory-safety-monitoring-syst...,0.93708,0.82124,0.88237,0.61816,None,305.570,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.959452,,,exp_B_online_aug,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,True
2,yolov8n.pt,yolov8n_C_offline_aug,trained,C:\Github\smart-factory-safety-monitoring-syst...,0.91579,0.75625,0.82045,0.55250,None,168.940,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.952494,,,exp_C_offline_aug,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,False
3,yolov8n.pt,yolov8n_D_full_pipeline,trained,C:\Github\smart-factory-safety-monitoring-syst...,0.93992,0.81936,0.87769,0.60914,None,413.738,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.959452,,,exp_D_full_pipeline,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,True


## 7. Benchmark Validation Latency

Latency is measured on a small subset of validation images for trained runs only. The untouched test set is not used for ranking or benchmarking here.


In [8]:
def validation_images_dir_from_yaml(yaml_path: Path) -> Path:
    with yaml_path.open("r", encoding="utf-8") as file_handle:
        data_config = yaml.safe_load(file_handle)
    dataset_root = (yaml_path.parent / data_config["path"]).resolve()
    return dataset_root / data_config["val"]


latency_rows = []
for _, row in ablation_results.iterrows():
    best_weights = Path(str(row.get("best_weights_path", "")))
    if row.get("status") != "trained" or not best_weights.exists():
        latency_rows.append(
            {
                "experiment": row.get("experiment"),
                "latency_status": "skipped",
                "benchmark_images": 0,
                "avg_inference_ms": None,
                "fps": None,
                "latency_error": "no trained best weights available",
            }
        )
        continue

    latency = benchmark_model_latency(
        weights_path=best_weights,
        sample_images_dir=validation_images_dir_from_yaml(Path(row["data_yaml"])),
        imgsz=int(training_config["imgsz"]),
        device=training_config["device"],
        max_images=int(training_config.get("latency_max_images", 20)),
    )
    latency["experiment"] = row.get("experiment")
    latency_rows.append(latency)

latency_df = pd.DataFrame(latency_rows)
ablation_results = ablation_results.merge(latency_df, on="experiment", how="left")
display(latency_df)

,latency_status,benchmark_images,avg_inference_ms,fps,latency_error,experiment
0,benchmarked,20,66.969450,14.932182,,exp_A_original_only
1,benchmarked,20,64.499645,15.503961,,exp_B_online_aug
2,benchmarked,20,65.267195,15.321633,,exp_C_offline_aug
3,benchmarked,20,61.751700,16.193886,,exp_D_full_pipeline


## 8. Save Reports

The report files separate complete results, failures, and the validation-based ranking. These CSV files are generated artifacts and should remain ignored by Git.


In [9]:
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

results_csv = REPORTS_DIR / "ablation_results.csv"
failures_csv = REPORTS_DIR / "ablation_failures.csv"
ranking_csv = REPORTS_DIR / "ablation_ranking.csv"

ablation_failures = ablation_results[ablation_results["status"].eq("failed")].copy()
ablation_ranking = rank_ablation_results(ablation_results)

ablation_results.to_csv(results_csv, index=False)
ablation_failures.to_csv(failures_csv, index=False)
ablation_ranking.to_csv(ranking_csv, index=False)

print(f"Saved: {results_csv.relative_to(V2_ROOT)}")
print(f"Saved: {failures_csv.relative_to(V2_ROOT)}")
print(f"Saved: {ranking_csv.relative_to(V2_ROOT)}")

display(
    ablation_ranking[
        [
            "rank",
            "experiment",
            "status",
            "recall",
            "map50",
            "map50_95",
            "fps",
            "rank_score",
            "uses_online_augmentation",
        ]
    ]
)

Saved: reports\training\ablation_results.csv
Saved: reports\training\ablation_failures.csv
Saved: reports\training\ablation_ranking.csv


,rank,experiment,status,recall,map50,map50_95,fps,rank_score,uses_online_augmentation
0,1,exp_B_online_aug,trained,0.82124,0.88237,0.61816,15.503961,8.221230e+08,True
1,2,exp_D_full_pipeline,trained,0.81936,0.87769,0.60914,16.193886,8.202383e+08,True
2,3,exp_A_original_only,trained,0.77549,0.83867,0.55291,14.932182,7.763292e+08,False
3,4,exp_C_offline_aug,trained,0.75625,0.82045,0.55250,15.321633,7.570710e+08,False


## 9. Optional Figures

The plots below summarize validation recall, mAP50, and validation-image latency when those metrics are available.


In [10]:
FIGURES_DIR = V2_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

try:
    import matplotlib.pyplot as plt

    plot_df = ablation_ranking.copy()
    if not plot_df.empty:
        for metric, filename, title in [
            ("map50", "ablation_map50.png", "Ablation mAP50"),
            ("recall", "ablation_recall.png", "Ablation Recall"),
            ("fps", "ablation_latency.png", "Ablation FPS"),
        ]:
            if metric in plot_df.columns and plot_df[metric].notna().any():
                ax = plot_df.plot.bar(
                    x="experiment", y=metric, legend=False, figsize=(8, 4)
                )
                ax.set_title(title)
                ax.set_xlabel("Experiment")
                ax.set_ylabel(metric)
                plt.xticks(rotation=25, ha="right")
                plt.tight_layout()
                out_path = FIGURES_DIR / filename
                plt.savefig(out_path, dpi=150)
                plt.close()
                print(f"Saved: {out_path.relative_to(V2_ROOT)}")
except Exception as exc:
    print(f"Figure generation skipped: {exc}")

Saved: reports\figures\ablation_map50.png
Saved: reports\figures\ablation_recall.png
Saved: reports\figures\ablation_latency.png


## 10. Checklist Before Notebook 08

- Confirm all four ablation runs completed or failed with clear logged reasons.
- Confirm ranking uses validation metrics only.
- Select the best training configuration from `ablation_ranking.csv`.
- Keep the test set untouched until final evaluation.
- Use Notebook 08 for final training/evaluation and export planning.
